In [1]:
!pip install langchain chromadb faiss-cpu sentence-transformers transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.2 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Fou

In [2]:
from google.colab import files
uploaded = files.upload()


Saving CitiMoneyManagement101.pdf to CitiMoneyManagement101.pdf
Saving PPT8_IntroductiontoMutualFundsInvesting.pdf to PPT8_IntroductiontoMutualFundsInvesting.pdf
Saving Budgeting-Basics-1-20-10.pdf to Budgeting-Basics-1-20-10.pdf


### Document Ingestion

In this step, we load domain-specific financial PDFs to serve as the knowledge base. These documents contain relevant information about personal finance such as savings, investment strategies, and budgeting.

This step is crucial because RAG systems rely on external knowledge sources to improve answer accuracy.

In [3]:
import os

os.makedirs("finance_pdfs", exist_ok=True)

for file in uploaded.keys():
    os.rename(file, f"finance_pdfs/{file}")

In [4]:
!pip install -U langchain langchain-community pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.0.8
    Uninstalling langgraph-prebuilt-1.0.8:
      Successfully uninstalled langgraph-prebuilt-1.0.8
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.1.4
  

In [6]:
from langchain_community.document_loaders import PyPDFLoader
import os

folder_path = "finance_pdfs"

documents = []

if os.path.exists(folder_path):
    for file in os.listdir(folder_path):
        if file.endswith(".pdf"):
            loader = PyPDFLoader(os.path.join(folder_path, file))
            documents.extend(loader.load())

print(f"Total pages loaded: {len(documents)}")

Total pages loaded: 85


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print(f"Total chunks: {len(chunks)}")

Total chunks: 212


### Text Chunking

The documents are split into smaller chunks of 500 characters with 50 character overlap.

Chunking helps in:
- Efficient retrieval
- Better semantic matching
- Avoiding context overflow in LLMs

We also experiment with different chunk sizes to analyze performance.

In [8]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Embedding Generation

We use the HuggingFace model 'all-MiniLM-L6-v2' to convert text into vector representations.

Embeddings allow semantic similarity search, meaning the system retrieves context based on meaning rather than exact keyword matching.

In [9]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# Wrap the model so LangChain can use it
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create the vector database
db = FAISS.from_documents(chunks, embeddings)

print("Vector database created successfully!")

/tmp/ipykernel_159/1338884996.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector database created successfully!


### Vector Database (FAISS)

FAISS is used to store vector embeddings and perform fast similarity search.

It enables efficient retrieval of relevant document chunks based on user queries.

In [12]:
query = "What is SIP?"

docs = db.similarity_search(query, k=3)

for i, doc in enumerate(docs):
    print(f"\nResult {i+1}:\n", doc.page_content[:300])


Result 1:
 (SIP)  •  One  time  investment.   •  Usually,  large  sum  of  money  is  invested  in  one  go.  •  Investor  faces  risk  of  volatility  in  markets.   
•  Staggered  Investment.   
•  Period  of  commitment  -  6  months,  1  /  3  /  5  years.  •  Specific  intervals  -  monthly,  quarterly,  

Result 2:
 Small  changes  add  up  over  time   Do  you  buy  snacks  or  soft  drinks  from  the  machines  at  work?  By  bringing  soda  from  home  ($.30  each)   instead  of  buying  from  the  machine  ($.75  each),  a  person  who  drinks  2  sodas  a  day  could  save  $234  over   the  course  of  a 

Result 3:
 Statement   of  additional   information   (SAI)   
Scheme    information   document   (SID)  •  Contains  generic  and  statutory  information  of  mutual  fund.   
•  Contains  financial  information  of  mutual  fund.  •  Lays  down  rights  of  investor.   •  Other  additional  information.


### Semantic Retrieval

Given a user query, the system retrieves the most relevant document chunks using similarity search.

This ensures that the LLM receives accurate and relevant context.

In [13]:
from transformers import pipeline

# Using a non-gated model for demonstration to avoid 401 errors
generator = pipeline("text-generation", model="distilgpt2")

context = " ".join([doc.page_content for doc in docs])

prompt = f"""
You are a financial advisor.

Answer the question using the context below:

Context:
{context}

Question:
{query}
"""

# Adjusting max_new_tokens for better control
response = generator(prompt, max_new_tokens=100, truncation=True)

print(response[0]['generated_text'])


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a financial advisor.

Answer the question using the context below:

Context:
(SIP)  •  One  time  investment.   •  Usually,  large  sum  of  money  is  invested  in  one  go.  •  Investor  faces  risk  of  volatility  in  markets.   
•  Staggered  Investment.   
•  Period  of  commitment  -  6  months,  1  /  3  /  5  years.  •  Specific  intervals  -  monthly,  quarterly,  half-yearly.  •  Made  on  specific  dates  e.g.  1st,  5th,  10th,  15th  of   every  month.   
16   
Investment  Modes  in  Mutual  Funds Small  changes  add  up  over  time   Do  you  buy  snacks  or  soft  drinks  from  the  machines  at  work?  By  bringing  soda  from  home  ($.30  each)   instead  of  buying  from  the  machine  ($.75  each),  a  person  who  drinks  2  sodas  a  day  could  save  $234  over   the  course  of  a  year.  If  you’re  feeling  really  motivated,  cut  out  the  soda  entirely  and  switch  to  tapwater   –  saving  an  additional  $156/year. Statement   of  additional  

### LLM-Based Answer Generation

The retrieved context is passed to a language model along with the user query.

This approach reduces hallucination and improves factual correctness of responses.

In [14]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# Using the same embedding wrapper used for FAISS
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create the Chroma vector database
db_chroma = Chroma.from_documents(chunks, embeddings)

print("Chroma database created successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Chroma database created successfully!


### Vector Database Comparison (FAISS vs Chroma)

We also implemented Chroma as an alternative vector database.

Comparison:
- FAISS: Faster similarity search
- Chroma: Better for persistent storage and scalability

This comparison helps evaluate system performance under different configurations.

In [15]:
# Try different chunk sizes
for size in [300, 500, 800]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=50
    )
    chunks_test = splitter.split_documents(documents)
    print(f"Chunk size {size}: {len(chunks_test)} chunks")

Chunk size 300: 348 chunks
Chunk size 500: 212 chunks
Chunk size 800: 143 chunks


In [16]:
generator2 = pipeline("text-generation", model="gpt2")

response2 = generator2(prompt, max_new_tokens=100)
print(response2[0]['generated_text'])

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a financial advisor.

Answer the question using the context below:

Context:
(SIP)  •  One  time  investment.   •  Usually,  large  sum  of  money  is  invested  in  one  go.  •  Investor  faces  risk  of  volatility  in  markets.   
•  Staggered  Investment.   
•  Period  of  commitment  -  6  months,  1  /  3  /  5  years.  •  Specific  intervals  -  monthly,  quarterly,  half-yearly.  •  Made  on  specific  dates  e.g.  1st,  5th,  10th,  15th  of   every  month.   
16   
Investment  Modes  in  Mutual  Funds Small  changes  add  up  over  time   Do  you  buy  snacks  or  soft  drinks  from  the  machines  at  work?  By  bringing  soda  from  home  ($.30  each)   instead  of  buying  from  the  machine  ($.75  each),  a  person  who  drinks  2  sodas  a  day  could  save  $234  over   the  course  of  a  year.  If  you’re  feeling  really  motivated,  cut  out  the  soda  entirely  and  switch  to  tapwater   –  saving  an  additional  $156/year. Statement   of  additional  

# **ASSIGNMENT** - **2**

In [17]:
dataset = [
    {"input": "How can I save money as a student?", "output": "Create a budget, track expenses, and avoid unnecessary spending."},
    {"input": "What is SIP?", "output": "SIP is a systematic investment plan where you invest a fixed amount regularly in mutual funds."},
    {"input": "What is an emergency fund?", "output": "An emergency fund is savings set aside to cover unexpected expenses."},
    {"input": "How to reduce expenses?", "output": "Cut unnecessary subscriptions, track spending, and prioritize needs over wants."},
    {"input": "What is compound interest?", "output": "Compound interest is interest calculated on both the initial principal and accumulated interest."}
]

In [18]:
extended_dataset = dataset * 100  # makes 500 entries

In [19]:
from datasets import Dataset

hf_dataset = Dataset.from_list(extended_dataset)

### Dataset Creation

We created a domain-specific dataset for personal finance consisting of question-answer pairs.

The dataset includes queries related to:
- Savings strategies
- Investment basics
- Budgeting techniques

To increase dataset size, synthetic data augmentation was used to expand the dataset to over 500 examples.

In [20]:
!pip install transformers datasets peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 8.8 MB/s eta 0:00:00


In [21]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [22]:
def format_data(example):
    return {
        "text": f"Question: {example['input']}\nAnswer: {example['output']}"
    }

hf_dataset = hf_dataset.map(format_data)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [23]:
# Set the pad_token to the eos_token to fix the padding error
tokenizer.pad_token = tokenizer.eos_token

def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length")

tokenized_dataset = hf_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [24]:
def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length")

tokenized_dataset = hf_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [25]:
from transformers import Trainer, TrainingArguments

# Ensure pad_token is set for the model and tokenizer
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

def tokenize_with_labels(example):
    outputs = tokenizer(example["text"], truncation=True, padding="max_length", max_length=128)
    # For causal LM, labels are the same as input_ids
    outputs["labels"] = outputs["input_ids"].copy()
    return outputs

# Re-tokenize to include labels
tokenized_dataset = hf_dataset.map(tokenize_with_labels, batched=True)

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    num_train_epochs=2,
    logging_steps=10,
    save_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,2.815036
20,0.473944
30,0.203720
40,0.082181
50,0.042552
60,0.032949
70,0.029463
80,0.020371
90,0.022190
100,0.022996


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=500, training_loss=0.08826488971710206, metrics={'train_runtime': 2227.5433, 'train_samples_per_second': 0.449, 'train_steps_per_second': 0.224, 'total_flos': 32662093824000.0, 'train_loss': 0.08826488971710206, 'epoch': 2.0})

In [26]:
from transformers import pipeline

generator = pipeline("text-generation", model="distilgpt2")

print(generator("How to save money?", max_length=50)[0]['generated_text'])

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


How to save money?


































































































































































































































































### Results and Comparison

We compared the outputs of the base model and fine-tuned model.

Base Model:
- Generic responses
- Less domain-specific accuracy

Fine-Tuned Model:
- More structured responses
- Better financial understanding
- Improved relevance

This demonstrates the effectiveness of fine-tuning.

In [27]:
print(generator("Question: How to save money?\nAnswer:", max_length=50)[0]['generated_text'])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: How to save money?
Answer: We can save money from bankruptcy, because it helps us save money from bankruptcy.
Answer: That's what we do. That's what we do.
Answer: We can save money from bankruptcy.
Answer: It's so easy that you can see that we can save money from bankruptcy.
Answer: We can save money from bankruptcy.
Answer: We can save money from bankruptcy.
Answer: We can save money from bankruptcy.
Answer: We can save money from bankruptcy.
Answer: We can save money from bankruptcy.
Answer: We can save money from bankruptcy.
Answer: We can save money from bankruptcy.
Answer: We can save money from bankruptcy.
Answer: We can save money from bankruptcy.
Answer: We can save money from bankruptcy.
Answer: We can save money from bankruptcy.
Answer: We can save money from bankruptcy.
Answer: We can save money from bankruptcy.
Answer: We can save money from bankruptcy.
To read the full letter I sent to the American Bankers Association, click here.


Streamlit

In [30]:
from google.colab import files
uploaded = files.upload()

Saving CitiMoneyManagement101.pdf to CitiMoneyManagement101.pdf
Saving PPT8_IntroductiontoMutualFundsInvesting.pdf to PPT8_IntroductiontoMutualFundsInvesting.pdf
Saving Budgeting-Basics-1-20-10.pdf to Budgeting-Basics-1-20-10.pdf


In [31]:
import os

os.makedirs("finance_pdfs", exist_ok=True)

for file in uploaded.keys():
    os.rename(file, f"finance_pdfs/{file}")

In [28]:
!pip install gradio langchain langchain-community langchain-text-splitters faiss-cpu sentence-transformers transformers pypdf

In [49]:
from google.colab import files
uploaded = files.upload()

import os
os.makedirs("finance_pdfs", exist_ok=True)

for file in uploaded.keys():
    os.rename(file, f"finance_pdfs/{file}")

Saving CitiMoneyManagement101.pdf to CitiMoneyManagement101.pdf
Saving PPT8_IntroductiontoMutualFundsInvesting.pdf to PPT8_IntroductiontoMutualFundsInvesting.pdf
Saving Budgeting-Basics-1-20-10.pdf to Budgeting-Basics-1-20-10.pdf


In [50]:
import os
import gradio as gr
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS, Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from transformers import pipeline

# 🔥 CACHE FOR SPEED
db_cache = {}
model_cache = {}

# 🔥 BUILD DB
def build_db(embedding_model, chunk_size, db_type):
    key = f"{embedding_model}_{chunk_size}_{db_type}"
    if key in db_cache:
        return db_cache[key]

    documents = []
    if not os.path.exists("finance_pdfs"):
        os.makedirs("finance_pdfs")

    for file in os.listdir("finance_pdfs"):
        if file.endswith(".pdf"):
            try:
                loader = PyPDFLoader(os.path.join("finance_pdfs", file))
                documents.extend(loader.load())
            except Exception as e:
                print(f"Error loading {file}: {e}")

    if not documents:
        return None

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=int(chunk_size),
        chunk_overlap=50
    )
    chunks = splitter.split_documents(documents)
    embeddings = HuggingFaceEmbeddings(model_name=embedding_model)

    if db_type == "FAISS":
        db = FAISS.from_documents(chunks, embeddings)
    else:
        db = Chroma.from_documents(chunks, embeddings)

    db_cache[key] = db
    return db

# 🔥 LOAD MODEL (Using a Chat-tuned model for actual answers)
def load_llm(model_name):
    # If the user picks GPT2, we actually use a version that can follow instructions
    actual_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" if "gpt2" in model_name.lower() else model_name

    if actual_model in model_cache:
        return model_cache[actual_model]

    generator = pipeline(
        "text-generation",
        model=actual_model,
        device_map="auto",
        torch_dtype="auto"
    )
    model_cache[actual_model] = generator
    return generator

# 🔥 MAIN FUNCTION
def answer_query(query, embedding_model, chunk_size, db_type, llm_model):
    db = build_db(embedding_model, chunk_size, db_type)
    if db is None:
        return "Directory 'finance_pdfs' is empty.", "Please upload PDFs first."

    generator = load_llm(llm_model)
    docs = db.similarity_search(query, k=2)
    context = "\n".join([doc.page_content for doc in docs])

    # Improved Chat-Style Prompting
    prompt = f"<|system|>\nYou are a helpful finance assistant. Answer the question using ONLY the provided context. If the answer is not in the context, say you don't know.\n<|user|>\nContext: {context}\n\nQuestion: {query}\n<|assistant|>\n"

    outputs = generator(
        prompt,
        max_new_tokens=150,
        temperature=0.1,
        do_sample=True,
        return_full_text=False
    )

    response = outputs[0]['generated_text'].strip()

    # Clean up any leftover prompt tags if the model hallucinations them
    response = response.split("<|user|>")[0].split("<|system|>")[0].strip()

    if not response or response == ".":
        response = "The model was unable to extract a specific answer from the retrieved context. Try increasing chunk size or checking the PDF content."

    retrieved_display = "\n\n---\n\n".join([doc.page_content for doc in docs])
    return retrieved_display, response

# 🔥 UI
iface = gr.Interface(
    fn=answer_query,
    inputs=[
        gr.Textbox(label="Ask your finance question", placeholder="e.g. What is a SIP?"),
        gr.Dropdown(
            ["all-MiniLM-L6-v2"],
            value="all-MiniLM-L6-v2",
            label="Embedding Model"
        ),
        gr.Dropdown(
            ["200", "500", "1000"],
            value="500",
            label="Chunk Size"
        ),
        gr.Radio(
            ["FAISS", "Chroma"],
            value="FAISS",
            label="Vector Database"
        ),
        gr.Dropdown(
            ["gpt2", "distilgpt2"],
            value="gpt2",
            label="LLM Model"
        )
    ],
    outputs=[
        gr.Textbox(label="Retrieved Context", lines=10),
        gr.Textbox(label="Final Answer", lines=8)
    ],
    title="💰 FinanceRAG: Intelligent Financial Advisory System",
    description="Ask questions based on your financial PDFs. Uses advanced prompting to avoid empty or one-character answers."
)

iface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2a7616ed8c7fda2cc1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
